# Finding Data in Mongoose

Queries are made with model methods like `find()`, `findOne()` and `findById()`.

## Common query methods

| Method | Returns | Notes |
|---|---|---|
| `Model.find(filter)` | Array of documents | Empty array `[]` when nothing matches — never `null` |
| `Model.findOne(filter)` | Single document | `null` when nothing matches |
| `Model.findById(id)` | Single document | Shorthand for `findOne({ _id: id })` |
| `Model.countDocuments(filter)` | Number | Counts without loading documents |
| `Model.exists(filter)` | `{ _id }` or `null` | Cheapest way to test existence |
| `Model.distinct(field, filter)` | Array of unique values | |

The difference in "not found" behaviour matters: `find()` gives a falsy-looking but truthy `[]`, so guard with `.length`, while `findOne()` and `findById()` give `null` and need a null check.

```javascript
const users = await User.find();          // [] if the collection is empty
const user  = await User.findById(id);
if (!user) return res.status(404).send('Not found');
```

## Examples

### Finding all documents

```javascript
const users = await User.find();
```

### Filtering results

Pass a query object to filter on field values:

```javascript
const tennisPlayers = await Athlete.find({ sport: 'Tennis' });
```

Multiple keys are combined with an implicit AND:

```javascript
await Movie.find({ rating: 'PG-13', year: 2012 });
```

### Chaining options

```javascript
const results = await Person.find({ age: { $gte: 18 } })
  .sort({ age: 1 })     //  1 ascending, -1 descending
  .limit(10)
  .skip(20)             // pagination offset
  .select('name age')   // projection: only these fields (plus _id)
  .lean();              // plain objects instead of hydrated documents
```

Projection syntax: `'name age'` includes, `'-password'` excludes. You can't mix inclusion and exclusion in one projection, except for excluding `_id`.

## Query operators

| Category | Operators |
|---|---|
| Comparison | `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`, `$in`, `$nin` |
| Logical | `$and`, `$or`, `$not`, `$nor` |
| Element | `$exists`, `$type` |
| Array | `$all`, `$size`, `$elemMatch` |
| Evaluation | `$regex`, `$expr`, `$text` |

```javascript
await Movie.find({ year: { $gte: 1990, $lte: 2005 } });
await Movie.find({ rating: { $in: ['PG', 'PG-13'] } });
await Movie.find({ $or: [{ score: { $gt: 8 } }, { rating: 'R' }] });
await Movie.find({ title: { $regex: /giant/i } });
await User.find({ deletedAt: { $exists: false } });
```

## Queries are thenables, not promises

`Model.find()` returns a `Query`, which only executes when you `await` it, call `.then()`, or call `.exec()`. That's what makes chaining possible — the query is being built up until the moment you await it.

```javascript
const query = Movie.find({ rating: 'R' }); // nothing has hit the database yet
query.sort({ year: -1 });
const docs = await query;                   // executes here
```

Two practical consequences:

- Awaiting the same query object twice throws in Mongoose 7+. Build a fresh query instead.
- `.exec()` gives you a real promise and better stack traces on errors. Prefer it in code where you'll be debugging failures: `await Movie.find({}).exec()`.

## `.lean()`

By default every result is a hydrated Mongoose document with getters, virtuals, `save()` and change tracking. That's overhead you don't need for a read-only endpoint.

```javascript
const movies = await Movie.find().lean();   // plain JS objects, notably faster
```

The trade-off: no `save()`, no virtuals, no instance methods, and `_id` is still an ObjectId rather than a string. Use it for anything you're serialising straight to JSON; skip it when you intend to modify and save.

## Common gotchas

- **`findById` with a malformed id throws.** A string that isn't a valid 24-character hex ObjectId raises a `CastError`, not a `null` result. Guard with `mongoose.Types.ObjectId.isValid(id)` or catch it in your error middleware — otherwise a bad URL param becomes a 500 instead of a 404.
- **Empty filter matches everything.** `find({})` and `deleteMany({})` are the same shape; the latter is how people accidentally empty a collection.
- **`sort` + `skip` on large collections is slow.** Without an index on the sort field, MongoDB sorts in memory and errors past 32 MB. Cursor-based pagination (`_id > lastSeen`) scales better than `skip`.
- **Undefined filter values are dropped.** `find({ name: undefined })` returns everything, because Mongoose strips undefined keys. If a query param can be undefined, build the filter conditionally.
- **Casting follows the schema.** `find({ year: '2012' })` works because Mongoose casts the string to a Number. A value it can't cast throws a `CastError`.
- **Strict query filtering.** Since Mongoose 7, `strictQuery` defaults to `false`, so filtering on a field not in your schema is passed through to MongoDB and simply matches nothing rather than being stripped.

## Related: reading queries in the shell

The equivalents in mongosh, useful for verifying what your app actually wrote:

```javascript
use movieApp          // database name — the last segment of your connection URI
show collections
db.movies.find()
db.movies.find({ rating: 'R' }).pretty()
db.movies.countDocuments()
```

## Sources

- [Mongoose queries](https://mongoosejs.com/docs/queries.html)
- [Query API](https://mongoosejs.com/docs/api/query.html)
- [MongoDB query operators](https://www.mongodb.com/docs/manual/reference/operator/query/)